#ASAP Summer School
Thursday 11/6/2026

##Hands on session on solar structure segmentation and solar flare forecasting

# Session 1: Segmentation using Basic Computer Vision Operaiotions (BCVO)

In this part, you will build a complete image-segmentation pipeline focused on the detection and characterization of solar active regions, thereby establishing a connection with solar flare forecasting. The workflow begins by partitioning a grayscale solar image into multiple intensity classes using Multi-Otsu thresholding. Subsequently, the resulting binary masks are refined through a series of morphological operations to remove noise, fill gaps, and improve the overall quality of the segmentation. The final segmented regions can then be used to extract descriptive features that may provide valuable information for active-region characterization and flare prediction.

### Fetch Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Registering an Email Address for Accessing JSOC Data with SunPy

The Joint Science Operations Center (JSOC) requires users to register and validate an email address before submitting data export requests. This registration is mandatory when downloading SDO/AIA, SDO/HMI, and other JSOC-hosted datasets through the SunPy API or the underlying DRMS library. Queries can be performed without registration, but downloading data requires a validated email address.

To register your email address:

1. Open the JSOC Email Registration page:

   http://jsoc.stanford.edu/ajax/register_email.html

2. Enter the email address you intend to use for JSOC requests.

3. Submit the registration form.

4. JSOC will send a confirmation email to the specified address.

5. Follow the instructions in the email to validate the address.

Once the email has been validated, it can be used in SunPy through the `Notify` attribute when creating a JSOC query.

Example:

```python
from sunpy.net import Fido, attrs as a
from astropy import units as u

result = Fido.search(
    a.Time("2016-02-13T00:00:00", "2016-02-13T00:10:00"),
    a.jsoc.Series("aia.lev1_euv_12s"),
    a.jsoc.Notify("your_email@example.com"),
    a.jsoc.Segment("image"),
    a.Wavelength(193*u.angstrom)
)
```

The email address specified in `a.jsoc.Notify()` must be the same validated address registered with JSOC. During the export process, JSOC uses this address to identify the requester and to manage data staging and download requests.

After registration, the email only needs to be validated once and can be reused for all future SunPy and DRMS download requests.


In [ ]:
from pathlib import Path

# this creates a nice structure for a general purpose project
PROJECT_DIR = Path("/content/drive/MyDrive/solar_segmentation")
RAW_DIR = PROJECT_DIR / "raw_fits"
PROC_DIR = PROJECT_DIR / "processed"
MODEL_DIR = PROJECT_DIR / "models"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_DIR)

We will now install the Python packages required to run the notebook in Google Colab.

In [ ]:
pip install sunpy zeep drms reproject mpl_animators


In [ ]:
from sunpy.net import Fido, attrs as a
from sunpy.net import jsoc
from astropy import units as u
from pathlib import Path

# Required by JSOC export service
JSOC_EMAIL = "panosgon@yahoo.gr" # enter your mail here

outdir = Path("jsoc_aia193_level1")
outdir.mkdir(exist_ok=True)

# Example: AIA 193 Å Level 1, 1 image every minute
query = Fido.search(
    a.Time("2012-05-12T15:00:00", "2012-05-12T15:10:00"),
    a.jsoc.Series("aia.lev1_euv_12s"),
    a.jsoc.Notify(JSOC_EMAIL),
    a.jsoc.Segment("image"),
    a.Wavelength(193 * u.angstrom),
    a.Sample(60 * u.second),
)

print(query)

files = Fido.fetch(
    query,
    path=str(outdir / "{file}"),
)

print(files)

You can now explore the downloaded FITS files. Select one of the files and provide its path in the variable `fits_file` below. This file will be used in the subsequent steps for visualization, preprocessing, and image analysis.

In [ ]:
import sunpy.map
import matplotlib.pyplot as plt

# Load as SunPy map
fits_file = "/content/drive/MyDrive/solar_segmentation/jsoc_aia193_level1/aia.lev1_euv_12s.2012-05-12T150009Z.193.image_lev1.fits"
aia_map = sunpy.map.Map(fits_file)

# Display image
plt.figure(figsize=(8, 8))
aia_map.plot(clip_interval=(1, 99.9) * u.percent)
plt.colorbar(label="Intensity")
plt.title("SDO/AIA 193 Å Level 1")
plt.show()

We now preprocess the image to make it suitable for machine learning applications.

Before applying segmentation or machine learning techniques, it is important to normalize the image intensities. Solar images often contain extreme bright and dark pixels caused by active regions, flares, coronal holes, instrumental effects, or noise. These outliers can dominate the image statistics and reduce the visibility of the majority of solar structures.

As a first step, the original SDO/AIA image is resized to 512 × 512 pixels, converted to a single-channel grayscale representation, and the pixel intensities are normalized.

Pixel-value normalization can be performed in several ways depending on the application. A straightforward approach is linear normalization, where the intensity values are rescaled to the range [0, 1]. This ensures that all input features are on a consistent scale, which typically improves the stability and convergence of machine learning models during training.

More advanced normalization techniques, such as percentile-based clipping or logarithmic scaling, can also be employed to better handle the large dynamic range commonly observed in solar EUV images. However, linear normalization provides a simple and effective starting point for many machine learning tasks.

If we opt for logarithmic scaling:

* Bright active regions become less dominant.
* Faint structures become more visible.
* Dynamic range is compressed.

Instead of using the absolute minimum and maximum pixel values, we compute the 1st and 99th intensity percentiles:

```python
p1, p99 = np.percentile(img, [1, 99])
```

The image intensities are then clipped to this range:

```python
img = np.clip(img, p1, p99)
```

This operation suppresses extreme outliers by assigning all values below the 1st percentile to `p1` and all values above the 99th percentile to `p99`.

Finally, the clipped image is linearly scaled to the range [0, 1]:

```python
img = (img - p1) / (p99 - p1 + 1e-8)
```

The small constant `1e-8` prevents division by zero in rare cases where the two percentiles are equal.



In [ ]:
import numpy as np
from sunpy.map import Map
from skimage.transform import resize
import matplotlib.pyplot as plt

# Load FITS
aia_map = Map(fits_file)

# Extract image
img = aia_map.data.astype(np.float32)

# Remove invalid values
img[~np.isfinite(img)] = 0

# Ensure positive values
img[img < 0] = 0

# Log transform
img = np.log1p(img)

# Robust normalization
p1, p99 = np.percentile(img, [1, 99])

img = np.clip(img, p1, p99)
img = (img - p1) / (p99 - p1 + 1e-8)

# Resize
img = resize(
    img,
    (512, 512),
    anti_aliasing=True,
    preserve_range=True
).astype(np.float32)

# Add channel dimension
# this step is not necessary right no
# typically done for deep learning pipelines

img_ml = np.expand_dims(img, axis=0) #

print("Shape:", img_ml.shape)
print("Min:", img_ml.min())
print("Max:", img_ml.max())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Original image
img_raw = aia_map.data.astype(np.float32)
img_raw[~np.isfinite(img_raw)] = 0
img_raw[img_raw < 0] = 0

# Log transform
img_log = np.log1p(img_raw)

# Normalization
p1, p99 = np.percentile(img_log, [1, 99])

img_norm = np.clip(img_log, p1, p99)
img_norm = (img_norm - p1) / (p99 - p1 + 1e-8)

# Histograms
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

ax[0].hist(img_raw.ravel(), bins=256)
ax[0].set_title("Original Intensities")

ax[1].hist(img_log.ravel(), bins=256)
ax[1].set_title("After log1p()")

ax[2].hist(img_norm.ravel(), bins=256)
ax[2].set_title("After Normalization [0,1]")

for a in ax:
    a.set_xlabel("Pixel Value")
    a.set_ylabel("Count")

plt.tight_layout()
plt.show()


print("Shape:", img_norm.shape)
print("Min:", img_norm.min())
print("Max:", img_norm.max())

### Explore an image

1.	Load the grayscale image.
2.	Display the image with matplotlib. Add a title and a colorbar.
3.	Plot the histogram of pixel intensities using plt.hist(). How many intensity modes (peaks) can you identify? What does each peak likely correspond to physically?
4.	Compute and print: min, max, mean, and standard deviation of pixel values.
5. Preprocess image and recalculate

Hint: image.ravel() flattens a 2D array into a 1D array suitable for histogram plotting.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sunpy.map import Map

# Load FITS image
aia_map = Map(fits_file)

# Extract image data
img = aia_map.data.astype(np.float32)

# Remove invalid values
img[~np.isfinite(img)] = 0
img[img < 0] = 0

# ---------------------------
# Display image
# ---------------------------
plt.figure(figsize=(8, 8))

im = plt.imshow(
    img,
    cmap="gray",
    origin="lower"
)

plt.title("SDO/AIA 193 Å Level 1 Image")
plt.colorbar(im, label="Intensity (DN)")
plt.show()

# ---------------------------
# Histogram
# ---------------------------
plt.figure(figsize=(8, 5))

plt.hist(
    img.ravel(),
    bins=256
)

plt.title("Histogram of Pixel Intensities")
plt.xlabel("Pixel Intensity (DN)")
plt.ylabel("Number of Pixels")

# Optional for solar images
plt.yscale("log")

plt.show()

# ---------------------------
# Statistics
# ---------------------------
print("Image Statistics")
print("----------------")
print(f"Min  : {img.min():.2f}")
print(f"Max  : {img.max():.2f}")
print(f"Mean : {img.mean():.2f}")
print(f"Std  : {img.std():.2f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sunpy.map import Map
from skimage.transform import resize

# =========================
# 1. Load image
# =========================

aia_map = Map(fits_file)

img_raw = aia_map.data.astype(np.float32)

# Clean invalid values
img_raw[~np.isfinite(img_raw)] = 0
img_raw[img_raw < 0] = 0

# =========================
# 2. Display original image
# =========================

vmin, vmax = np.percentile(img_raw, [1, 99.5])

plt.figure(figsize=(8, 8))
im = plt.imshow(
    img_raw,
    cmap="gray",
    origin="lower",
    vmin=vmin,
    vmax=vmax
)
plt.title("Original SDO/AIA 193 Å Image")
plt.colorbar(im, label="Intensity")
plt.show()

# =========================
# 3. Histogram of original image
# =========================

plt.figure(figsize=(8, 5))
plt.hist(img_raw.ravel(), bins=256, log=True)
plt.title("Histogram of Original Pixel Intensities")
plt.xlabel("Pixel Intensity")
plt.ylabel("Number of Pixels (log scale)")
plt.show()

# =========================
# 4. Original image statistics
# =========================

print("Original image statistics")
print("-------------------------")
print(f"Min  : {img_raw.min():.4f}")
print(f"Max  : {img_raw.max():.4f}")
print(f"Mean : {img_raw.mean():.4f}")
print(f"Std  : {img_raw.std():.4f}")

# =========================
# 5. Preprocess image
# =========================

# Resize to 512 x 512
img_resized = resize(
    img_raw,
    (512, 512),
    anti_aliasing=True,
    preserve_range=True
).astype(np.float32)

# Linear percentile normalization to [0, 1]
p1, p99 = np.percentile(img_resized, [1, 99.9])

img_preprocessed = np.clip(img_resized, p1, p99)
img_preprocessed = (img_preprocessed - p1) / (p99 - p1 + 1e-8)

# =========================
# 6. Display preprocessed image
# =========================

plt.figure(figsize=(8, 8))
im = plt.imshow(
    img_preprocessed,
    cmap="gray",
    origin="lower",
    vmin=0,
    vmax=1
)
plt.title("Preprocessed Image: 512 × 512, Normalized [0, 1]")
plt.colorbar(im, label="Normalized intensity")
plt.show()

# =========================
# 7. Histogram of preprocessed image
# =========================

plt.figure(figsize=(8, 5))
plt.hist(img_preprocessed.ravel(), bins=256)
plt.title("Histogram of Preprocessed Pixel Intensities")
plt.xlabel("Normalized Pixel Intensity")
plt.ylabel("Number of Pixels")
plt.show()

# =========================
# 8. Preprocessed image statistics
# =========================

print("\nPreprocessed image statistics")
print("-----------------------------")
print(f"Min  : {img_preprocessed.min():.4f}")
print(f"Max  : {img_preprocessed.max():.4f}")
print(f"Mean : {img_preprocessed.mean():.4f}")
print(f"Std  : {img_preprocessed.std():.4f}")

# =========================
# 9. Optional: side-by-side comparison
# =========================

fig, ax = plt.subplots(2, 2, figsize=(14, 12))

ax[0, 0].imshow(img_raw, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
ax[0, 0].set_title("Original Image")
ax[0, 0].axis("off")

ax[0, 1].hist(img_raw.ravel(), bins=256, log=True)
ax[0, 1].set_title("Original Histogram")
ax[0, 1].set_xlabel("Intensity")
ax[0, 1].set_ylabel("Count (log scale)")

ax[1, 0].imshow(img_preprocessed, cmap="gray", origin="lower", vmin=0, vmax=1)
ax[1, 0].set_title("Preprocessed Image")
ax[1, 0].axis("off")

ax[1, 1].hist(img_preprocessed.ravel(), bins=256)
ax[1, 1].set_title("Preprocessed Histogram")
ax[1, 1].set_xlabel("Normalized intensity")
ax[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.show()

### Otsu thresholding

5.	Apply standard single-threshold Otsu using skimage.filters.threshold_otsu(). Create a binary mask and display it alongside the original.
6.	What threshold value was found? Mark it on your histogram from previous assignment
7.	Visually assess the quality of the segmentation. Does a single threshold work well for this image?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu

# Use the preprocessed image from before
# img_preprocessed should be 2D, normalized in [0, 1]

img = img_preprocessed

# =========================
# 1. Apply Otsu threshold
# =========================

otsu_threshold = threshold_otsu(img)

binary_mask = img > otsu_threshold

print(f"Otsu threshold value: {otsu_threshold:.4f}")

# =========================
# 2. Display original + mask
# =========================

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

im0 = ax[0].imshow(img, cmap="gray", origin="lower", vmin=0, vmax=1)
ax[0].set_title("Preprocessed AIA 193 Å Image")
ax[0].axis("off")
plt.colorbar(im0, ax=ax[0], fraction=0.046)

im1 = ax[1].imshow(binary_mask, cmap="gray", origin="lower")
ax[1].set_title("Binary Mask from Otsu Threshold")
ax[1].axis("off")
plt.colorbar(im1, ax=ax[1], fraction=0.046)

plt.tight_layout()
plt.show()

# =========================
# 3. Histogram with threshold
# =========================

plt.figure(figsize=(8, 5))

plt.hist(
    img.ravel(),
    bins=256
)

plt.axvline(
    otsu_threshold,
    linestyle="--",
    linewidth=2,
    label=f"Otsu threshold = {otsu_threshold:.4f}"
)

plt.title("Histogram with Otsu Threshold")
plt.xlabel("Normalized pixel intensity")
plt.ylabel("Number of pixels")
plt.legend()
plt.show()

### Multi-Otsu

8.	Apply Multi-Otsu thresholding with skimage.filters.threshold_multiotsu(image, classes=3).
9.	Use np.digitize() to assign each pixel to a class (0, 1, 2, …). Visualise the segmented image using a discrete colormap (e.g., plt.cm.Set1).
10.	Experiment: re-run with classes=4. Which value of classes gives the most meaningful segmentation for your image?
11.	Mark the multi-Otsu thresholds on your histogram. How do they compare to the single-Otsu threshold?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu, threshold_multiotsu
from matplotlib.colors import BoundaryNorm

# Use preprocessed image in [0, 1]
img = img_preprocessed

# =========================
# Single Otsu
# =========================

otsu_thr = threshold_otsu(img)
mask_otsu = img > otsu_thr

print(f"Single Otsu threshold: {otsu_thr:.4f}")

# =========================
# Multi-Otsu: 3 classes
# =========================

thresholds_3 = threshold_multiotsu(img, classes=3)
regions_3 = np.digitize(img, bins=thresholds_3)

print("Multi-Otsu thresholds, 3 classes:", thresholds_3)

# =========================
# Multi-Otsu: 4 classes
# =========================

thresholds_4 = threshold_multiotsu(img, classes=4)
regions_4 = np.digitize(img, bins=thresholds_4)

print("Multi-Otsu thresholds, 4 classes:", thresholds_4)

# =========================
# Visualize segmentations
# =========================

fig, ax = plt.subplots(1, 4, figsize=(22, 5))

ax[0].imshow(img, cmap="gray", origin="lower", vmin=0, vmax=1)
ax[0].set_title("Preprocessed Image")
ax[0].axis("off")

ax[1].imshow(mask_otsu, cmap="gray", origin="lower")
ax[1].set_title("Single Otsu\n2 classes")
ax[1].axis("off")

im2 = ax[2].imshow(regions_3, cmap=plt.cm.Set1, origin="lower", vmin=0, vmax=2)
ax[2].set_title("Multi-Otsu\n3 classes")
ax[2].axis("off")

im3 = ax[3].imshow(regions_4, cmap=plt.cm.Set1, origin="lower", vmin=0, vmax=3)
ax[3].set_title("Multi-Otsu\n4 classes")
ax[3].axis("off")

plt.tight_layout()
plt.show()

# =========================
# Histogram with all thresholds
# =========================

plt.figure(figsize=(10, 6))

plt.hist(
    img.ravel(),
    bins=256,
    alpha=0.8
)

plt.axvline(
    otsu_thr,
    linestyle="--",
    linewidth=2,
    label=f"Single Otsu = {otsu_thr:.4f}"
)

for i, t in enumerate(thresholds_3):
    plt.axvline(
        t,
        linestyle="-",
        linewidth=2,
        label=f"Multi-Otsu 3 classes T{i+1} = {t:.4f}"
    )

for i, t in enumerate(thresholds_4):
    plt.axvline(
        t,
        linestyle=":",
        linewidth=2,
        label=f"Multi-Otsu 4 classes T{i+1} = {t:.4f}"
    )

plt.title("Histogram with Single-Otsu and Multi-Otsu Thresholds")
plt.xlabel("Normalized pixel intensity")
plt.ylabel("Number of pixels")
plt.legend()
plt.show()

# =========================
# Optional: class pixel fractions
# =========================

def print_class_fractions(regions, name):
    print(f"\n{name}")
    print("-" * len(name))
    total = regions.size
    for cls in np.unique(regions):
        count = np.sum(regions == cls)
        fraction = 100 * count / total
        print(f"Class {cls}: {fraction:.2f}% of pixels")

print_class_fractions(regions_3, "Multi-Otsu 3 classes")
print_class_fractions(regions_4, "Multi-Otsu 4 classes")

With 3 classes:
Class 0 usually corresponds to dark structures, including coronal holes and low-intensity background.
Class 1 usually corresponds to quiet Sun / average corona.
Class 2 usually corresponds to bright structures, such as active regions and bright coronal loops.

With 4 classes:
Class 0 may better isolate the darkest pixels.
Class 1 may represent quiet-Sun background.
Class 2 may represent moderately bright coronal structures.
Class 3 may isolate the brightest active-region cores or flare-related emission.

For AIA 193 Å, 3 classes is often a good first choice because it roughly separates dark, medium, and bright solar structures.
However, 4 classes can be more meaningful if the image contains strong active regions, because it separates very bright active-region cores from moderately bright loop structures.

Compared to single Otsu, Multi-Otsu places multiple thresholds across the intensity distribution.
Single Otsu only creates a dark/bright split, while Multi-Otsu can represent multiple physical populations in the solar image.

### Extracting a Binary Mask for One Region
12.	From your multi-Otsu result, isolate the foreground region (the brightest class) as a binary mask.
13.	Display the mask. What imperfections do you observe? Consider: noise, holes, ragged edges, isolated small blobs.


In [ ]:
thresholds = threshold_multiotsu(img, classes=3)
regions = np.digitize(img, bins=thresholds)

In [ ]:
regions == 2

# because the classes are numbered:

# 0 -> dark pixels
# 1 -> medium intensity pixels
# 2 -> brightest pixels

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Brightest class
bright_mask = (regions == 2)

print("Mask shape:", bright_mask.shape)
print("Foreground pixels:", np.sum(bright_mask))

plt.figure(figsize=(8,8))
plt.imshow(bright_mask, cmap="gray", origin="lower")
plt.title("Brightest Multi-Otsu Class")
plt.axis("off")
plt.show()

### Morphological Filtering
Apply each of the following morphological operations to the binary mask from Exercise 1.4. Display all results in a single figure (one subplot per operation).


1.   Erosion (Remove thin protrusions):	skimage.morphology.erosion

2.   Dilation (Fill small holes)	skimage.morphology.dilation
3.   Opening (Remove noise (erode then dilate))	skimage.morphology.opening
4.   Closing (Fill gaps (dilate then erode))	skimage.morphology.closing


For each operation:


*   Use a disk structuring element of radius 3: skimage.morphology.disk(3)
*   Display the result next to the original mask.
*   Briefly describe in one sentence what changed.

Hint: Try radius = 1, 3, and 5. How does the structuring element size affect the result?


In [ ]:
import matplotlib.pyplot as plt
from skimage.morphology import erosion, dilation, opening, closing, disk

# Binary mask from previous exercise
# Example:
# bright_mask = (regions == 2)

mask = bright_mask

# Structuring element
selem = disk(3)

# Apply morphological operations
mask_erosion = erosion(mask, selem)
mask_dilation = dilation(mask, selem)
mask_opening = opening(mask, selem)
mask_closing = closing(mask, selem)

# Display all results
fig, ax = plt.subplots(1, 5, figsize=(22, 5))

ax[0].imshow(mask, cmap="gray", origin="lower")
ax[0].set_title("Original Mask")
ax[0].axis("off")

ax[1].imshow(mask_erosion, cmap="gray", origin="lower")
ax[1].set_title("Erosion")
ax[1].axis("off")

ax[2].imshow(mask_dilation, cmap="gray", origin="lower")
ax[2].set_title("Dilation")
ax[2].axis("off")

ax[3].imshow(mask_opening, cmap="gray", origin="lower")
ax[3].set_title("Opening")
ax[3].axis("off")

ax[4].imshow(mask_closing, cmap="gray", origin="lower")
ax[4].set_title("Closing")
ax[4].axis("off")

plt.tight_layout()
plt.show()

### Build the pipeline of the BCVO algorithm

14.	Build a clean pipeline: Multi-Otsu → binary mask → Opening → Closing.
15.	Count connected components before and after filtering using skimage.measure.label() and skimage.measure.regionprops().
16.	Compute the area (in pixels) of each connected region. Plot a histogram of region areas.
17.	Filter out regions smaller than 100 pixels. How many objects remain?
18.	(Bonus) Overlay the final segmentation contours on the original image using skimage.segmentation.find_boundaries().


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from skimage.filters import threshold_multiotsu
from skimage.morphology import opening, closing, disk, remove_small_objects
from skimage.measure import label, regionprops
from skimage.segmentation import find_boundaries

# Use your preprocessed image in [0, 1]
img = img_preprocessed

# =========================
# 1. Multi-Otsu → binary mask
# =========================

thresholds = threshold_multiotsu(img, classes=3)
regions = np.digitize(img, bins=thresholds)

# Brightest class as foreground
binary_mask = regions == 2

# =========================
# 2. Opening → Closing
# =========================

selem = disk(3)

mask_opened = opening(binary_mask, selem)
mask_filtered = closing(mask_opened, selem)

# =========================
# 3. Connected components before/after
# =========================

labels_before = label(binary_mask)
props_before = regionprops(labels_before)

labels_after = label(mask_filtered)
props_after = regionprops(labels_after)

areas_before = [p.area for p in props_before]
areas_after = [p.area for p in props_after]

print("Before filtering")
print("----------------")
print(f"Number of objects: {len(props_before)}")

print("\nAfter opening + closing")
print("-----------------------")
print(f"Number of objects: {len(props_after)}")

# =========================
# 4. Area histogram
# =========================

plt.figure(figsize=(10, 5))
plt.hist(areas_before, bins=50, alpha=0.6, label="Before filtering")
plt.hist(areas_after, bins=50, alpha=0.6, label="After opening + closing")
plt.xlabel("Region area [pixels]")
plt.ylabel("Number of regions")
plt.title("Histogram of Connected-Component Areas")
plt.legend()
plt.yscale("log")
plt.show()

# =========================
# 5. Remove regions smaller than 100 pixels
# =========================

final_mask = remove_small_objects(
    mask_filtered.astype(bool),
    min_size=100
)

labels_final = label(final_mask)
props_final = regionprops(labels_final)

areas_final = [p.area for p in props_final]

print("\nAfter removing objects < 100 pixels")
print("-----------------------------------")
print(f"Number of objects remaining: {len(props_final)}")

# =========================
# 6. Visualize pipeline
# =========================

fig, ax = plt.subplots(1, 5, figsize=(24, 5))

ax[0].imshow(img, cmap="gray", origin="lower")
ax[0].set_title("Preprocessed Image")
ax[0].axis("off")

ax[1].imshow(regions, cmap=plt.cm.Set1, origin="lower")
ax[1].set_title("Multi-Otsu Classes")
ax[1].axis("off")

ax[2].imshow(binary_mask, cmap="gray", origin="lower")
ax[2].set_title(f"Binary Mask\nObjects: {len(props_before)}")
ax[2].axis("off")

ax[3].imshow(mask_filtered, cmap="gray", origin="lower")
ax[3].set_title(f"Opening + Closing\nObjects: {len(props_after)}")
ax[3].axis("off")

ax[4].imshow(final_mask, cmap="gray", origin="lower")
ax[4].set_title(f"Final Mask\nObjects: {len(props_final)}")
ax[4].axis("off")

plt.tight_layout()
plt.show()

# =========================
# 7. Histogram after small-object removal
# =========================

plt.figure(figsize=(8, 5))
plt.hist(areas_final, bins=30)
plt.xlabel("Region area [pixels]")
plt.ylabel("Number of regions")
plt.title("Final Region Area Distribution")
plt.yscale("log")
plt.show()

# =========================
# 8. Bonus: overlay final contours
# =========================

boundaries = find_boundaries(final_mask, mode="outer")

plt.figure(figsize=(8, 8))

plt.imshow(img, cmap="gray", origin="lower", vmin=0, vmax=1)

# Overlay boundaries
overlay = np.ma.masked_where(~boundaries, boundaries)
plt.imshow(overlay, cmap="autumn", origin="lower", alpha=0.9)

plt.title("Final Segmentation Contours on AIA 193 Å Image")
plt.axis("off")
plt.show()

Based on these segmentation masks, we can extract a set of hand-crafted image features that may complement standard SHARP parameters for active-region characterization. These features describe the morphology, intensity distribution, and spatial structure of segmented regions, such as their area, compactness, elongation, brightness, and boundary complexity. When combined with SHARP magnetic-field descriptors, they can provide additional information about the coronal appearance and evolution of active regions, which may be useful for solar flare forecasting.

In [ ]:
import numpy as np
import pandas as pd
from skimage.measure import label, regionprops, regionprops_table
from skimage.segmentation import find_boundaries

# Inputs
# final_mask: binary segmentation mask
# img: preprocessed AIA 193 image in [0, 1]

labels = label(final_mask)

props = regionprops_table(
    labels,
    intensity_image=img,
    properties=[
        "label",
        "area",
        "bbox",
        "centroid",
        "eccentricity",
        "equivalent_diameter_area",
        "extent",
        "feret_diameter_max",
        "major_axis_length",
        "minor_axis_length",
        "orientation",
        "perimeter",
        "solidity",
        "mean_intensity",
        "max_intensity",
        "min_intensity",
    ]
)

features_df = pd.DataFrame(props)

# Add custom features
features_df["compactness"] = (
    4 * np.pi * features_df["area"] /
    (features_df["perimeter"]**2 + 1e-8)
)

features_df["elongation"] = (
    features_df["major_axis_length"] /
    (features_df["minor_axis_length"] + 1e-8)
)

features_df["intensity_contrast"] = (
    features_df["max_intensity"] -
    features_df["min_intensity"]
)

features_df["bbox_width"] = (
    features_df["bbox-3"] - features_df["bbox-1"]
)

features_df["bbox_height"] = (
    features_df["bbox-2"] - features_df["bbox-0"]
)

features_df["bbox_aspect_ratio"] = (
    features_df["bbox_width"] /
    (features_df["bbox_height"] + 1e-8)
)

features_df.head()

In [ ]:
features_df.to_csv("active_region_handcrafted_features.csv", index=False)

In [ ]:
global_features = {
    "n_regions": len(features_df),
    "total_area": features_df["area"].sum(),
    "mean_area": features_df["area"].mean(),
    "max_area": features_df["area"].max(),
    "mean_intensity_global": img[final_mask].mean(),
    "max_intensity_global": img[final_mask].max(),
    "coverage_fraction": final_mask.sum() / final_mask.size,
}

global_features_df = pd.DataFrame([global_features])
global_features_df